# Tabular Ensemble

`XGBoost raw + XGBoost residual48 + XGBoost correction7d + LightGBM raw`? ???? ??????.

- `XGBoost raw`: `target` ?? ??
- `XGBoost residual48`: `target - lag_48` ?? ? ??, ??? `installed_capacity` ???
- `XGBoost correction7d`: `target - same_hour_mean_7d` ?? ? ??, ??? `installed_capacity` ???
- `LightGBM raw`: `target` ?? ??

? ??? `split + global`?? ????, ???? 4? ??? ?? ??????.

In [1]:
from pathlib import Path

import itertools
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

In [2]:
DATA_DIR = Path('../data/processed_data')
TRAIN_PATH = DATA_DIR / 'train_processed.parquet'
VALID_PATH = DATA_DIR / 'valid_processed.parquet'
OUTPUT_PATH = Path('./tabular_ensemble_valid_predictions.csv')

TARGET_COL = 'target'
ROW_ID_COL = 'row_id'
BASELINE_COL = 'lag_48'
CAT_COLS = ['county', 'is_business', 'product_type', 'is_consumption', 'segment']
DROP_COLS = {TARGET_COL, ROW_ID_COL, 'datetime', 'date'}

train_df = pd.read_parquet(TRAIN_PATH)
valid_df = pd.read_parquet(VALID_PATH)

for col in CAT_COLS:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype('category')
        valid_df[col] = valid_df[col].astype('category')

FEATURE_COLS_ALL = [col for col in train_df.columns if col not in DROP_COLS]
FEATURE_COLS_SPLIT = [col for col in FEATURE_COLS_ALL if col != 'is_consumption']

print('train shape:', train_df.shape)
print('valid shape:', valid_df.shape)
print('feature count (all):', len(FEATURE_COLS_ALL))
print('feature count (split):', len(FEATURE_COLS_SPLIT))
print('baseline missing rate(train):', train_df[BASELINE_COL].isna().mean())
print('baseline missing rate(valid):', valid_df[BASELINE_COL].isna().mean())
print('same_hour_mean_7d missing rate(train):', train_df['same_hour_mean_7d'].isna().mean())
print('same_hour_mean_7d missing rate(valid):', valid_df['same_hour_mean_7d'].isna().mean())

train shape: (1614612, 140)
valid shape: (403740, 140)
feature count (all): 136
feature count (split): 135
baseline missing rate(train): 0.0
baseline missing rate(valid): 0.0
same_hour_mean_7d missing rate(train): 0.0
same_hour_mean_7d missing rate(valid): 0.0


In [3]:
def split_by_consumption(df):
    cons = df[df['is_consumption'] == 1].copy()
    prod = df[df['is_consumption'] == 0].copy()
    return cons, prod


def prepare_target(train_part, valid_part, target_mode, segment_name):
    if target_mode == 'raw':
        baseline_train = pd.Series(np.zeros(len(train_part)), index=train_part.index, dtype='float64')
        baseline_valid = pd.Series(np.zeros(len(valid_part)), index=valid_part.index, dtype='float64')
        scale_train = pd.Series(np.ones(len(train_part)), index=train_part.index, dtype='float64')
        scale_valid = pd.Series(np.ones(len(valid_part)), index=valid_part.index, dtype='float64')
        y_train = train_part[TARGET_COL].copy()
    elif target_mode == 'residual_48h':
        baseline_train = train_part[BASELINE_COL].fillna(0).astype('float64')
        baseline_valid = valid_part[BASELINE_COL].fillna(0).astype('float64')
        if segment_name == 'production':
            scale_train = train_part['installed_capacity'].fillna(1000).clip(lower=1).astype('float64')
            scale_valid = valid_part['installed_capacity'].fillna(1000).clip(lower=1).astype('float64')
            y_train = (train_part[TARGET_COL] - baseline_train) / scale_train
        else:
            scale_train = pd.Series(np.ones(len(train_part)), index=train_part.index, dtype='float64')
            scale_valid = pd.Series(np.ones(len(valid_part)), index=valid_part.index, dtype='float64')
            y_train = train_part[TARGET_COL] - baseline_train
    elif target_mode == 'correction_7d':
        baseline_train = train_part['same_hour_mean_7d'].fillna(0).astype('float64')
        baseline_valid = valid_part['same_hour_mean_7d'].fillna(0).astype('float64')
        if segment_name == 'production':
            scale_train = train_part['installed_capacity'].fillna(1000).clip(lower=1).astype('float64')
            scale_valid = valid_part['installed_capacity'].fillna(1000).clip(lower=1).astype('float64')
            y_train = (train_part[TARGET_COL] - baseline_train) / scale_train
        else:
            scale_train = pd.Series(np.ones(len(train_part)), index=train_part.index, dtype='float64')
            scale_valid = pd.Series(np.ones(len(valid_part)), index=valid_part.index, dtype='float64')
            y_train = train_part[TARGET_COL] - baseline_train
    else:
        raise ValueError(f'Unknown target_mode: {target_mode}')

    y_valid = valid_part[TARGET_COL].copy()
    y_eval = (y_valid - baseline_valid) / scale_valid
    return y_train, y_valid, y_eval, baseline_valid, scale_valid


def combine_with_baseline(pred_raw, baseline_valid, scale_valid, target_mode):
    if target_mode == 'raw':
        pred = pred_raw
    elif target_mode in {'residual_48h', 'correction_7d'}:
        pred = baseline_valid.to_numpy() + pred_raw * scale_valid.to_numpy()
    else:
        raise ValueError(f'Unknown target_mode: {target_mode}')
    return np.clip(pred, 0, None)


def align_prediction_frames(pred_split_df, pred_all_df, pred_col):
    pred_split_df = pred_split_df.sort_values(ROW_ID_COL).reset_index(drop=True)
    pred_all_df = pred_all_df.sort_values(ROW_ID_COL).set_index(ROW_ID_COL)
    pred_all_aligned = pred_all_df.loc[pred_split_df[ROW_ID_COL]].reset_index()
    pred_split_df[f'{pred_col}_split_global'] = 0.5 * pred_split_df[pred_col].values + 0.5 * pred_all_aligned[pred_col].values
    return pred_split_df


def summarize_segment_mae(mae_prod, mae_cons, mae_split, mae_all, mae_ensemble, model_name):
    return pd.DataFrame(
        {'MAE': [mae_prod, mae_cons, mae_split, mae_all, mae_ensemble]},
        index=[
            f'{model_name} production',
            f'{model_name} consumption',
            f'{model_name} overall(split)',
            f'{model_name} overall(single)',
            f'{model_name} split+global',
        ],
    )


def train_split_global(train_df, valid_df, feature_cols_split, feature_cols_all, trainer, pred_col, model_name, target_mode):
    tr_cons, tr_prod = split_by_consumption(train_df)
    va_cons, va_prod = split_by_consumption(valid_df)

    model_prod, pred_prod_df, mae_prod = trainer(tr_prod, va_prod, feature_cols_split, 'production', target_mode)
    model_cons, pred_cons_df, mae_cons = trainer(tr_cons, va_cons, feature_cols_split, 'consumption', target_mode)
    model_all, pred_all_df, mae_all = trainer(train_df, valid_df, feature_cols_all, 'overall', target_mode)

    pred_split_df = pd.concat([pred_prod_df, pred_cons_df], axis=0).sort_values(ROW_ID_COL).reset_index(drop=True)
    mae_split = mean_absolute_error(pred_split_df[TARGET_COL], pred_split_df[pred_col])

    pred_split_df = align_prediction_frames(pred_split_df, pred_all_df, pred_col)
    ensemble_col = f'{pred_col}_split_global'
    mae_ensemble = mean_absolute_error(pred_split_df[TARGET_COL], pred_split_df[ensemble_col])

    summary = summarize_segment_mae(mae_prod, mae_cons, mae_split, mae_all, mae_ensemble, model_name)
    display(summary.style.format('{:.4f}'))

    return {
        'model_prod': model_prod,
        'model_cons': model_cons,
        'model_all': model_all,
        'pred_split_df': pred_split_df,
        'pred_all_df': pred_all_df,
        'summary': summary,
        'mae_prod': mae_prod,
        'mae_cons': mae_cons,
        'mae_split': mae_split,
        'mae_all': mae_all,
        'mae_ensemble': mae_ensemble,
        'pred_col': pred_col,
        'ensemble_col': ensemble_col,
        'target_mode': target_mode,
        'model_name': model_name,
    }

In [7]:
XGB_BASE_PARAMS = {
    'n_estimators': 5000,
    'learning_rate': 0.04,
    'max_depth': 7,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'device': 'cuda',
    'enable_categorical': True,
    'random_state': 42,
    'early_stopping_rounds': 50,
    'eval_metric': 'mae',
    'verbosity': 0,
}


def train_xgb_segment(train_part, valid_part, feature_cols, segment_name, target_mode):
    params = XGB_BASE_PARAMS.copy()
    model = xgb.XGBRegressor(**params)

    X_train = train_part[feature_cols].fillna(0)
    X_valid = valid_part[feature_cols].fillna(0)
    y_train, y_valid, y_eval, baseline_valid, scale_valid = prepare_target(train_part, valid_part, target_mode, segment_name)

    try:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_eval)], verbose=100)
    except xgb.core.XGBoostError as exc:
        print(f'[XGBoost:{segment_name}:{target_mode}] CUDA unavailable, falling back to CPU: {exc}')
        params['device'] = 'cpu'
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_eval)], verbose=100)

    pred_raw = model.predict(X_valid)
    pred = combine_with_baseline(pred_raw, baseline_valid, scale_valid, target_mode)
    pred_df = valid_part[[ROW_ID_COL, TARGET_COL]].copy()
    pred_df['pred_xgb'] = pred
    mae = mean_absolute_error(y_valid, pred)
    best_iter = getattr(model, 'best_iteration', None)
    print(f'[XGBoost:{segment_name}:{target_mode}] MAE = {mae:.4f} | Best iter: {best_iter}')
    return model, pred_df, mae

In [5]:
LGBM_BASE_PARAMS = {
    'objective': 'regression',
    'learning_rate': 0.04,
    'n_estimators': 1800,
    'num_leaves': 127,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'random_state': 42,
    'device_type': 'gpu',
}


def train_lgbm_segment(train_part, valid_part, feature_cols, segment_name, target_mode):
    params = LGBM_BASE_PARAMS.copy()
    model = lgb.LGBMRegressor(**params)

    X_train = train_part[feature_cols].copy().fillna(0)
    X_valid = valid_part[feature_cols].copy().fillna(0)
    y_train, y_valid, y_eval, baseline_valid, scale_valid = prepare_target(train_part, valid_part, target_mode, segment_name)
    categorical_cols = [col for col in CAT_COLS if col in feature_cols]

    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_eval)],
            eval_metric='mae',
            categorical_feature=categorical_cols,
            callbacks=[lgb.early_stopping(120), lgb.log_evaluation(100)],
        )
    except Exception as exc:
        print(f'[LightGBM:{segment_name}:{target_mode}] GPU unavailable, falling back to CPU: {exc}')
        params['device_type'] = 'cpu'
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_eval)],
            eval_metric='mae',
            categorical_feature=categorical_cols,
            callbacks=[lgb.early_stopping(120), lgb.log_evaluation(100)],
        )

    pred_raw = model.predict(X_valid)
    pred = combine_with_baseline(pred_raw, baseline_valid, scale_valid, target_mode)
    pred_df = valid_part[[ROW_ID_COL, TARGET_COL]].copy()
    pred_df['pred_lgbm'] = pred
    mae = mean_absolute_error(y_valid, pred)
    print(f'[LightGBM:{segment_name}:{target_mode}] MAE = {mae:.4f}')
    return model, pred_df, mae

In [8]:
xgb_raw_result = train_split_global(
    train_df,
    valid_df,
    FEATURE_COLS_SPLIT,
    FEATURE_COLS_ALL,
    train_xgb_segment,
    'pred_xgb',
    'XGBoost raw',
    'raw',
)

xgb_residual_result = train_split_global(
    train_df,
    valid_df,
    FEATURE_COLS_SPLIT,
    FEATURE_COLS_ALL,
    train_xgb_segment,
    'pred_xgb',
    'XGBoost residual48',
    'residual_48h',
)

xgb_correction7d_result = train_split_global(
    train_df,
    valid_df,
    FEATURE_COLS_SPLIT,
    FEATURE_COLS_ALL,
    train_xgb_segment,
    'pred_xgb',
    'XGBoost correction7d',
    'correction_7d',
)

[0]	validation_0-mae:179.22001
[100]	validation_0-mae:22.34729
[200]	validation_0-mae:19.63298
[300]	validation_0-mae:18.96350
[400]	validation_0-mae:18.57910
[500]	validation_0-mae:18.31776
[600]	validation_0-mae:18.13449
[700]	validation_0-mae:18.00015
[800]	validation_0-mae:17.90151
[900]	validation_0-mae:17.81027
[1000]	validation_0-mae:17.74039
[1100]	validation_0-mae:17.69193
[1200]	validation_0-mae:17.63043
[1300]	validation_0-mae:17.58133
[1400]	validation_0-mae:17.54376
[1500]	validation_0-mae:17.51052
[1600]	validation_0-mae:17.48030
[1700]	validation_0-mae:17.44087
[1800]	validation_0-mae:17.40877
[1900]	validation_0-mae:17.38079
[2000]	validation_0-mae:17.35678
[2100]	validation_0-mae:17.33374
[2200]	validation_0-mae:17.31440
[2300]	validation_0-mae:17.29524
[2400]	validation_0-mae:17.27955
[2500]	validation_0-mae:17.26927
[2600]	validation_0-mae:17.25527
[2700]	validation_0-mae:17.24268
[2800]	validation_0-mae:17.23190
[2900]	validation_0-mae:17.22145
[3000]	validation_0-m

,MAE
XGBoost raw production,17.1076
XGBoost raw consumption,15.6168
XGBoost raw overall(split),16.3622
XGBoost raw overall(single),14.0157
XGBoost raw split+global,13.5191


[0]	validation_0-mae:0.04554
[100]	validation_0-mae:0.01871
[200]	validation_0-mae:0.01594
[300]	validation_0-mae:0.01474
[400]	validation_0-mae:0.01410
[500]	validation_0-mae:0.01364
[600]	validation_0-mae:0.01333
[700]	validation_0-mae:0.01306
[800]	validation_0-mae:0.01286
[900]	validation_0-mae:0.01271
[1000]	validation_0-mae:0.01257
[1100]	validation_0-mae:0.01246
[1200]	validation_0-mae:0.01237
[1300]	validation_0-mae:0.01229
[1400]	validation_0-mae:0.01222
[1500]	validation_0-mae:0.01215
[1600]	validation_0-mae:0.01210
[1700]	validation_0-mae:0.01205
[1800]	validation_0-mae:0.01200
[1900]	validation_0-mae:0.01195
[2000]	validation_0-mae:0.01192
[2100]	validation_0-mae:0.01189
[2200]	validation_0-mae:0.01185
[2300]	validation_0-mae:0.01182
[2400]	validation_0-mae:0.01180
[2500]	validation_0-mae:0.01178
[2600]	validation_0-mae:0.01176
[2700]	validation_0-mae:0.01173
[2800]	validation_0-mae:0.01171
[2900]	validation_0-mae:0.01170
[3000]	validation_0-mae:0.01168
[3100]	validation_0-

,MAE
XGBoost residual48 production,15.7926
XGBoost residual48 consumption,14.6292
XGBoost residual48 overall(split),15.2109
XGBoost residual48 overall(single),14.1824
XGBoost residual48 split+global,12.7382


[0]	validation_0-mae:0.03633
[100]	validation_0-mae:0.02556
[200]	validation_0-mae:0.02358
[300]	validation_0-mae:0.02270
[400]	validation_0-mae:0.02210
[500]	validation_0-mae:0.02170
[600]	validation_0-mae:0.02138
[700]	validation_0-mae:0.02115
[800]	validation_0-mae:0.02097
[900]	validation_0-mae:0.02083
[1000]	validation_0-mae:0.02069
[1100]	validation_0-mae:0.02057
[1200]	validation_0-mae:0.02049
[1300]	validation_0-mae:0.02038
[1400]	validation_0-mae:0.02032
[1500]	validation_0-mae:0.02026
[1600]	validation_0-mae:0.02021
[1700]	validation_0-mae:0.02015
[1800]	validation_0-mae:0.02007
[1900]	validation_0-mae:0.02001
[2000]	validation_0-mae:0.01998
[2100]	validation_0-mae:0.01995
[2200]	validation_0-mae:0.01990
[2300]	validation_0-mae:0.01987
[2400]	validation_0-mae:0.01985
[2500]	validation_0-mae:0.01982
[2600]	validation_0-mae:0.01980
[2700]	validation_0-mae:0.01978
[2800]	validation_0-mae:0.01976
[2900]	validation_0-mae:0.01975
[3000]	validation_0-mae:0.01973
[3100]	validation_0-

,MAE
XGBoost correction7d production,17.3902
XGBoost correction7d consumption,15.7443
XGBoost correction7d overall(split),16.5672
XGBoost correction7d overall(single),14.3714
XGBoost correction7d split+global,13.7208


In [9]:
lgbm_raw_result = train_split_global(
    train_df,
    valid_df,
    FEATURE_COLS_SPLIT,
    FEATURE_COLS_ALL,
    train_lgbm_segment,
    'pred_lgbm',
    'LightGBM raw',
    'raw',
)

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 25707
[LightGBM] [Info] Number of data points in the train set: 807306, number of used features: 133
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4060 Ti, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 122 dense feature groups (95.47 MB) transferred to GPU in 0.037213 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 71.726586
Training until validation scores don't improve for 120 rounds
[100]	valid_0's l1: 21.1645	valid_0's l2: 27106.3
[200]	valid_0's l1: 17.9007	valid_0's l2: 25103.5
[300]	valid_0's l1: 17.2023	valid_0's l2: 24985.6
[400]	valid_0's l1: 16.8689	valid_0's l2: 24935.3
[500]	valid_0's l1: 16.6827	valid_0's l2: 24942.9
Early stopping, best iteration is:
[417]	valid_0's l1: 16.826	valid_0's l2: 24924.6
[LightGBM:produ

,MAE
LightGBM raw production,16.8113
LightGBM raw consumption,17.4728
LightGBM raw overall(split),17.1421
LightGBM raw overall(single),16.5773
LightGBM raw split+global,14.9601


In [10]:
variant_df = (
    xgb_raw_result['pred_split_df'][[ROW_ID_COL, TARGET_COL, xgb_raw_result['ensemble_col']]]
    .rename(columns={xgb_raw_result['ensemble_col']: 'pred_xgb_raw'})
    .merge(
        xgb_residual_result['pred_split_df'][[ROW_ID_COL, xgb_residual_result['ensemble_col']]].rename(columns={xgb_residual_result['ensemble_col']: 'pred_xgb_residual'}),
        on=ROW_ID_COL,
        how='inner',
    )
    .merge(
        xgb_correction7d_result['pred_split_df'][[ROW_ID_COL, xgb_correction7d_result['ensemble_col']]].rename(columns={xgb_correction7d_result['ensemble_col']: 'pred_xgb_correction7d'}),
        on=ROW_ID_COL,
        how='inner',
    )
    .merge(
        lgbm_raw_result['pred_split_df'][[ROW_ID_COL, lgbm_raw_result['ensemble_col']]].rename(columns={lgbm_raw_result['ensemble_col']: 'pred_lgbm_raw'}),
        on=ROW_ID_COL,
        how='inner',
    )
    .sort_values(ROW_ID_COL)
    .reset_index(drop=True)
)

variant_df['pred_simple_avg'] = variant_df[['pred_xgb_raw', 'pred_xgb_residual', 'pred_xgb_correction7d', 'pred_lgbm_raw']].mean(axis=1)
simple_avg_mae = mean_absolute_error(variant_df[TARGET_COL], variant_df['pred_simple_avg'])

summary_df = pd.DataFrame(
    {'MAE': [
        xgb_raw_result['mae_ensemble'],
        xgb_residual_result['mae_ensemble'],
        xgb_correction7d_result['mae_ensemble'],
        lgbm_raw_result['mae_ensemble'],
        simple_avg_mae,
    ]},
    index=[
        'XGBoost raw split+global',
        'XGBoost residual48 split+global',
        'XGBoost correction7d split+global',
        'LightGBM raw split+global',
        'Simple average (4 variants)',
    ],
)

display(summary_df.style.format('{:.4f}'))
variant_df.head()

,MAE
XGBoost raw split+global,13.5191
XGBoost residual48 split+global,12.7382
XGBoost correction7d split+global,13.7208
LightGBM raw split+global,14.9601
Simple average (4 variants),11.2814


,row_id,target,pred_xgb_raw,pred_xgb_residual,pred_xgb_correction7d,pred_lgbm_raw,pred_simple_avg
0,1614612,0.005,0.169273,0.340412,0.066395,0.601346,0.294357
1,1614613,726.559,727.954102,726.560493,721.112507,733.579453,727.301639
2,1614614,0.000,0.000000,0.001626,0.004371,0.000000,0.001499
3,1614615,29.203,28.176651,29.116234,29.286271,28.831492,28.852662
4,1614616,1.175,1.206997,0.415835,1.165937,0.797778,0.896637


In [11]:
weight_rows = []
weights = np.round(np.arange(0.0, 1.01, 0.1), 2)

for wxr, wxd, wxc in itertools.product(weights, weights, weights):
    wlr = round(1.0 - wxr - wxd - wxc, 2)
    if wlr < 0 or wlr > 1:
        continue
    pred = (
        wxr * variant_df['pred_xgb_raw']
        + wxd * variant_df['pred_xgb_residual']
        + wxc * variant_df['pred_xgb_correction7d']
        + wlr * variant_df['pred_lgbm_raw']
    )
    mae = mean_absolute_error(variant_df[TARGET_COL], pred)
    weight_rows.append({
        'weight_xgb_raw': wxr,
        'weight_xgb_residual': wxd,
        'weight_xgb_correction7d': wxc,
        'weight_lgbm_raw': wlr,
        'mae': mae,
    })

weight_table = pd.DataFrame(weight_rows).sort_values('mae').reset_index(drop=True)
display(weight_table.head(20).style.format({
    'weight_xgb_raw': '{:.2f}',
    'weight_xgb_residual': '{:.2f}',
    'weight_xgb_correction7d': '{:.2f}',
    'weight_lgbm_raw': '{:.2f}',
    'mae': '{:.4f}',
}))

best_weights = weight_table.iloc[0]
print('Best weights:')
print(best_weights)

,weight_xgb_raw,weight_xgb_residual,weight_xgb_correction7d,weight_lgbm_raw,mae
0,0.20,0.40,0.30,0.10,10.9988
1,0.30,0.40,0.30,-0.00,11.0033
2,0.20,0.50,0.20,0.10,11.0376
3,0.30,0.40,0.20,0.10,11.0384
4,0.30,0.50,0.20,-0.00,11.0419
5,0.20,0.50,0.30,0.00,11.0547
6,0.40,0.40,0.20,-0.00,11.0599
7,0.10,0.50,0.30,0.10,11.0670
8,0.30,0.30,0.30,0.10,11.0680
9,0.10,0.40,0.30,0.20,11.0716


Best weights:
weight_xgb_raw              0.200000
weight_xgb_residual         0.400000
weight_xgb_correction7d     0.300000
weight_lgbm_raw             0.100000
mae                        10.998758
Name: 0, dtype: float64


In [12]:
best_pred = (
    best_weights['weight_xgb_raw'] * variant_df['pred_xgb_raw']
    + best_weights['weight_xgb_residual'] * variant_df['pred_xgb_residual']
    + best_weights['weight_xgb_correction7d'] * variant_df['pred_xgb_correction7d']
    + best_weights['weight_lgbm_raw'] * variant_df['pred_lgbm_raw']
)
variant_df['pred_best_weighted'] = best_pred
best_weighted_mae = mean_absolute_error(variant_df[TARGET_COL], variant_df['pred_best_weighted'])

final_summary = summary_df.copy()
final_summary.loc['Best weighted ensemble'] = best_weighted_mae
display(final_summary.style.format('{:.4f}'))

variant_df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved validation predictions to: {OUTPUT_PATH.resolve()}')

,MAE
XGBoost raw split+global,13.5191
XGBoost residual48 split+global,12.7382
XGBoost correction7d split+global,13.7208
LightGBM raw split+global,14.9601
Simple average (4 variants),11.2814
Best weighted ensemble,10.9988


Saved validation predictions to: C:\Users\AN\Desktop\kaggle\notebooks\tabular_ensemble_valid_predictions.csv
